# B7 · Clip and select by location: inside, touching, cut to shape

> **Time:** about 90 minutes  
> **You need:** Part A and Lessons B1 to B6. You should be able to filter a GeoDataFrame with a mask, make shapely points (B2), and draw layers with `ax=ax`.  
> **By the end you can:**
> 1. use the **spatial predicates** `within`, `contains`, `intersects` and `touches` to ask yes-or-no questions about shapes,
> 2. select features by location, such as the districts that touch Kathmandu or the landslides inside Dolakha,
> 3. cut layers to a box or a province with `gpd.clip`, and explain the difference between selecting and clipping.

In B5 you chose districts by the values in their table. Many questions are about **where** things are instead. Which districts border Kathmandu? Which landslides lie inside Dolakha? What part of each district falls inside the Kathmandu Valley? This lesson answers such questions with the shapes themselves.

**Counting lines in this lesson:** the lines that only set up a plot, `fig, ax = plt.subplots()` and `plt.show()`, are not counted in the Q1/Q2/Q3 line counts.

**How to practise:** after every step there are three questions. **Q1** needs one line of code, **Q2** two lines, **Q3** about three. Type your answer in the empty cell under each question, run it, then open **Show solution** to check. Try first, even if you are not sure: struggling for a few minutes is how the skill grows.

<!-- pygile-nav -->
**PyGILE Course** · lesson 17 of 21 · [Course home](../README.md)  
← Previous: [B6 · Joins and dissolve: names, mismatches and provinces](B06_Joins_and_dissolve.ipynb) · Next: [B8 · Buffers and nearest features: how close is close?](B08_Buffers_and_nearest_features.ipynb) →

## 1. Spatial predicates: yes-or-no questions about shapes

A **spatial predicate** is a question about two shapes whose answer is `True` or `False`. You met three of them in B2. Here they are with a fourth, `touches`:

| Code | `True` when… |
| --- | --- |
| `a.within(b)` | `a` lies completely inside `b` |
| `a.contains(b)` | `b` lies completely inside `a` (the reverse of `within`) |
| `a.intersects(b)` | `a` and `b` share at least one point: they overlap, touch, or one is inside the other |
| `a.touches(b)` | they share points on their edges, but their insides do not overlap |

The example uses four made-up squares, in made-up units, to make the rules visible. `box(x_min, y_min, x_max, y_max)`, which you met in B2, makes a rectangle from its corners, with x first as always.

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from shapely.geometry import Point, box
from pathlib import Path

big = box(0, 0, 4, 4)
small = box(1, 1, 2, 2)    # inside big
side = box(4, 1, 6, 3)     # shares part of an edge with big
corner = box(3, 3, 5, 5)   # overlaps one corner of big

fig, ax = plt.subplots(figsize=(5, 4))
gpd.GeoSeries([big, small, side, corner]).plot(
    ax=ax, facecolor="none", edgecolor=["black", "tab:blue", "tab:orange", "tab:green"], linewidth=2)
for label, x, y in [("big", 0.2, 3.5), ("small", 1.1, 1.4), ("side", 4.6, 1.9), ("corner", 4.1, 4.5)]:
    ax.text(x, y, label)
ax.set_title("Four made-up squares")
plt.show()

In [ ]:
rows = []
for name, shape in [("small", small), ("side", side), ("corner", corner)]:
    rows.append({"shape": name,
                 "within big": shape.within(big),
                 "intersects big": shape.intersects(big),
                 "touches big": shape.touches(big)})
pd.DataFrame(rows)

Read the table row by row, with the picture beside it:

- `small` is **within** `big`. It also intersects it, but does not touch it: touching means meeting **only** at the edges.
- `side` meets `big` along an edge and nothing more, so it **touches** and **intersects**, but is not within.
- `corner` overlaps part of `big`. It **intersects**, but it neither touches (the insides overlap) nor is within (part of it is outside).

`intersects` is the broadest question: it is `True` whenever the two shapes meet in any way.

### Practice

**Q1 · one line:** Does `big` contain `small`? Answer with one predicate.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
big.contains(small)
```

`True`. `big.contains(small)` asks the same question as `small.within(big)`, from the other side.
</details>

**Q2 · two lines:** Make a fourth square `far = box(7, 7, 8, 8)`. Print whether it intersects `big`, and whether it touches `big`.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
far = box(7, 7, 8, 8)
print(far.intersects(big), far.touches(big))
```

Both are `False`: `far` shares no point at all with `big`. Shapes that do not intersect are called **disjoint**.
</details>

**Q3 · three lines:** A point exactly on the edge of `big` is `Point(4, 2)`. Print whether it is within `big`, whether it touches `big`, and whether it intersects `big`.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
edge_point = Point(4, 2)
print(edge_point.within(big))
print(edge_point.touches(big))
print(edge_point.intersects(big))
```

`False`, `True`, `True`. A point on the boundary is **not** within the shape: `within` means inside, not on the edge. `intersects` counts edge points, so it is the safer question when points may lie on a border.
</details>

## 2. Asking every district at once

On a GeoDataFrame, a predicate asks the question of **every row**. `districts.contains(point)` checks each of the 77 districts against one point, and gives a Series of 77 `True`/`False` values.

That Series is a boolean mask, as in B5. Put it in square brackets to keep the districts where the answer is `True`.

Below, the eight peaks from `nepal_peaks.csv` become points, as in B3. Then each peak is matched to the district that contains it.

The loop uses `zip(a, b)`, which walks through two sequences side by side and gives one pair per round. It is like `enumerate` from A3, which gives a position and a value. Here each pair is a peak's name and its point.

**Predict:** every peak is in Nepal, so every peak should find exactly one district. Will it?

In [ ]:
DATA = Path("..") / "data"
districts = gpd.read_file(DATA / "nepal_districts.geojson")
provinces = gpd.read_file(DATA / "nepal_provinces.geojson")
hq = gpd.read_file(DATA / "district_headquarters.geojson")

peaks_table = pd.read_csv(DATA / "nepal_peaks.csv")
peaks = gpd.GeoDataFrame(peaks_table,
                         geometry=gpd.points_from_xy(peaks_table["lon"], peaks_table["lat"]),
                         crs="EPSG:4326")

lhotse = peaks.loc[peaks["name"] == "Lhotse", "geometry"].iloc[0]
districts[districts.contains(lhotse)]

In [ ]:
for name, point in zip(peaks["name"], peaks.geometry):
    found = districts.loc[districts.contains(point), "district"]
    print(name, list(found))

Lhotse is in Solukhumbu. `.loc[mask, "geometry"].iloc[0]` takes one shapely shape out of the table, so it can be used in a predicate.

The loop holds a surprise. Five peaks find their district, but **Everest, Makalu and Cho Oyu find none**: their lists are empty. These summits sit on Nepal's border with China. The peak coordinates are approximate, and the boundaries are simplified to about 300 m, so a summit point can land a little outside the polygon.

This is a real lesson about data. Points on or near a border need care: check them, and never assume that every point falls inside some polygon. In B8 you will measure how far each of these three points lies from its nearest district.

### Practice

**Q1 · one line:** Show the district that contains Kanchenjunga. Take its point from `peaks` first, as was done for Lhotse.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
districts[districts.contains(peaks.loc[peaks["name"] == "Kanchenjunga", "geometry"].iloc[0])]
```

Kanchenjunga is in Taplejung, in the far east. It fits in one line, but storing the point in a variable first, as in the example, is easier to read.
</details>

**Q2 · two lines:** Take the point of the Kathmandu district headquarters from `hq` (its `name` is `Kathmandu`). Print the district that contains it.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
ktm_hq = hq.loc[hq["name"] == "Kathmandu", "geometry"].iloc[0]
print(districts.loc[districts.contains(ktm_hq), "district"])
```

The headquarters point lies in Kathmandu district. Note that `hq` has only a `name` column; the district of each headquarters has to be found by location, as here.
</details>

**Q3 · three lines:** Count how many districts contain each peak, using `.sum()` on the mask inside a loop. Print each peak's name and its count.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
for name, point in zip(peaks["name"], peaks.geometry):
    n_found = districts.contains(point).sum()
    print(name, n_found)
```

Five peaks give 1 and three give 0. A count of 2 would be a warning too: it would mean two districts overlap. Counting matches is a quick check on any point-in-polygon question.
</details>

## 3. Neighbours: the districts that touch Kathmandu

`touches` finds **neighbours**: shapes that share a border but do not overlap.

First take Kathmandu's shape out of the table with `.loc[mask, "geometry"].iloc[0]`. Then `districts.touches(kathmandu)` asks every district whether it touches Kathmandu.

`touches` works here because neighbouring districts in this file share **exactly** the same border lines. With layers from different sources, borders often have tiny gaps or overlaps, and `touches` can miss real neighbours. Then `intersects` is the safer choice.

`intersects` also returns Kathmandu itself, because every shape intersects itself.

**Predict:** how many districts border Kathmandu?

In [ ]:
kathmandu = districts.loc[districts["district"] == "Kathmandu", "geometry"].iloc[0]
neighbours = districts[districts.touches(kathmandu)]
print(len(neighbours))
print(list(neighbours["district"]))
print(districts.intersects(kathmandu).sum())

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
neighbours.plot(ax=ax, color="tab:orange", edgecolor="white")
districts[districts["district"] == "Kathmandu"].plot(ax=ax, color="tab:red", edgecolor="white")
ax.set_title("Kathmandu (red) and the districts that touch it")
plt.show()

Seven districts touch Kathmandu: Bhaktapur, Dhading, Kabhrepalanchok, Lalitpur, Makawanpur, Nuwakot and Sindhupalchok. `intersects` finds 8, because it also counts Kathmandu itself.

The map shows only Kathmandu and its neighbours, so it zooms in on them automatically.

### Practice

**Q1 · one line:** How many districts touch Chitawan? Take its shape first, as for Kathmandu.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(districts.touches(districts.loc[districts["district"] == "Chitawan", "geometry"].iloc[0]).sum())
```

Chitawan has 6 neighbours. Remember the spelling of this file: Chitawan.
</details>

**Q2 · two lines:** List the neighbours of Mustang.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
mustang = districts.loc[districts["district"] == "Mustang", "geometry"].iloc[0]
print(list(districts.loc[districts.touches(mustang), "district"]))
```

Mustang touches only 3 districts in Nepal: Dolpa, Manang and Myagdi. Its long northern border is with China, which is not in this layer.
</details>

**Q3 · three lines:** Find the neighbours of Dolpa that are in a **different** province from Dolpa (`NP06`). Combine a predicate mask and an attribute mask with `&`, and print their names and province codes.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
dolpa = districts.loc[districts["district"] == "Dolpa", "geometry"].iloc[0]
mask = districts.touches(dolpa) & (districts["province_code"] != "NP06")
print(districts.loc[mask, ["district", "province_code"]])
```

Three neighbours are in other provinces: Mustang and Myagdi in Gandaki (`NP04`), and Rukum East in Lumbini (`NP05`). Location masks and attribute masks are both Series of `True`/`False`, so `&` combines them.
</details>

## 4. Points inside a polygon: landslides in Dolakha

The reverse question is as common: which **points** lie inside one polygon? `slides.within(dolakha)` asks each landslide whether it lies inside Dolakha.

The landslide CSV also names a district for each record. Comparing the two answers is a good **cross-check**. When the coordinates and the names agree, you can trust both a little more.

The setup cell below applies the Chitwan → Chitawan fix from B6, so the names match the map.

In [ ]:
slides_table = pd.read_csv(DATA / "practice_landslides.csv")
slides = gpd.GeoDataFrame(slides_table,
                          geometry=gpd.points_from_xy(slides_table["lon"], slides_table["lat"]),
                          crs="EPSG:4326")
slides["district"] = slides["district"].replace({"Chitwan": "Chitawan"})   # the fix from B6

dolakha = districts.loc[districts["district"] == "Dolakha", "geometry"].iloc[0]
in_dolakha = slides[slides.within(dolakha)]
print(len(in_dolakha))
print(in_dolakha["district"].value_counts())

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
districts.plot(ax=ax, color="lightgrey", edgecolor="white")
districts[districts["district"] == "Dolakha"].plot(ax=ax, color="tab:green")
slides.plot(ax=ax, color="grey", markersize=8)
in_dolakha.plot(ax=ax, color="black", markersize=12)
ax.set_title("Practice landslides inside Dolakha (black)")
plt.show()

9 landslides lie inside Dolakha, and all 9 are labelled Dolakha in the CSV. The location and the names agree.

On the map the selected points are drawn last, in black, so they sit on top of the grey ones.

### Practice

**Q1 · one line:** How many landslides lie within Chitawan?

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(slides.within(districts.loc[districts["district"] == "Chitawan", "geometry"].iloc[0]).sum())
```

9, the same as the number of records labelled Chitawan after the fix.
</details>

**Q2 · two lines:** Take the shape of Gandaki province (`NP04`) from `provinces`, then count the landslides within it.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
gandaki = provinces.loc[provinces["province_code"] == "NP04", "geometry"].iloc[0]
print(slides.within(gandaki).sum())
```

10 landslides are in Gandaki. A province is a polygon like any other, so the same predicate works.
</details>

**Q3 · three lines:** Loop over the district names `["Dolakha", "Rasuwa", "Sindhupalchok", "Gorkha"]`. For each, take its shape and print the name and the number of landslides within it.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
for name in ["Dolakha", "Rasuwa", "Sindhupalchok", "Gorkha"]:
    shape = districts.loc[districts["district"] == name, "geometry"].iloc[0]
    print(name, slides.within(shape).sum())
```

Dolakha 9, Rasuwa 5, Sindhupalchok 3 and Gorkha 4: exactly the counts of the `district` column in A9 and B6. Looping over names is fine for a few districts; B9 counts points in all 77 districts at once with a spatial join.
</details>

## 5. Clipping: cutting a layer with a cookie-cutter

Selecting keeps or drops whole features. **Clipping** cuts them: `gpd.clip(layer, mask)` keeps only the parts of `layer` that fall inside `mask`, like a cookie-cutter pressed into dough.

- **Points** are kept or dropped, as in a selection, because a point cannot be cut.
- **Lines and polygons** are cut at the edge of the mask, so you get new, smaller shapes.

The mask can be a shapely shape, such as a `box`, or a GeoDataFrame, such as a province. The layer and the mask must use the **same CRS**.

Below, the approximate Kathmandu Valley box `valley_box` from B2 is the mask.

In [ ]:
valley_box = box(85.2, 27.55, 85.55, 27.82)   # Kathmandu Valley, approximate

valley_districts = gpd.clip(districts, valley_box)
valley_hq = gpd.clip(hq, valley_box)
print(len(valley_districts), list(valley_districts["district"]))
print(list(valley_hq["name"]))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
valley_districts.plot(ax=ax, column="district", legend=True, edgecolor="white",
                      legend_kwds={"loc": "upper left", "bbox_to_anchor": (1, 1)})
valley_hq.plot(ax=ax, color="black", markersize=20)
ax.set_title("Districts and headquarters clipped to the Kathmandu Valley box")
plt.show()

8 districts have some part inside the box, and each is cut at the box edge. Only 3 headquarters fall inside it: Lalitpur, Kathmandu and Bhaktapur.

The map is a neat rectangle: everything outside the box is gone. `legend_kwds` places the legend outside the map, on the right, so it does not cover the districts.

### Practice

**Q1 · one line:** Clip the landslides `slides` to `valley_box` and print how many are left.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(len(gpd.clip(slides, valley_box)))
```

0: none of the practice landslides fall inside the valley box. An empty result is a valid answer, not an error.
</details>

**Q2 · two lines:** Convert `valley_districts` to UTM 45N and print their total area in km². Then print the area of the box itself, using `gpd.GeoSeries([valley_box], crs="EPSG:4326")`.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(valley_districts.to_crs(32645).area.sum() / 1000000)
print(gpd.GeoSeries([valley_box], crs="EPSG:4326").to_crs(32645).area / 1000000)
```

The two results, about 1,032.91 and 1,032.85 km², agree to within 0.1 km². The clipped pieces fill the box completely, so their areas add up to the box's area. The tiny difference comes from how the box's edges are drawn after conversion. A shapely shape has no CRS, so you wrap it in a GeoSeries and state its CRS before converting.
</details>

**Q3 · three lines:** Make a box around the Everest region, `box(86.5, 27.6, 87.2, 28.1)`. Clip the districts to it, print the names of the districts inside, and draw them.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
everest_box = box(86.5, 27.6, 87.2, 28.1)
everest_districts = gpd.clip(districts, everest_box)
print(list(everest_districts["district"]))
everest_districts.plot(column="district", legend=True, edgecolor="white")
plt.show()
```

Four districts have parts in this box: Ramechhap, Sankhuwasabha, Solukhumbu and Dolakha, with Solukhumbu filling most of it. Notice how the clipped map has a straight edge on every side where the box cut through Nepal.
</details>

## 6. Clipping to a real shape: a province, and Nepal's neighbours

A mask can also be a GeoDataFrame, such as one province selected from `provinces`.

There is one trap when you clip polygons with a polygon that shares their borders. The neighbouring districts **touch** the mask along its edge. Their "part inside" is a thin line along the border, not an area, so `clip` returns those lines too.

`keep_geom_type=True` keeps only results of the same type as the input: polygons stay, the stray lines are dropped. Check `.geom_type.value_counts()` after any clip of polygons.

Clipping also makes **location maps**. Clip the countries of the world to a box around Nepal, and you get Nepal with its neighbours.

In [ ]:
gandaki_gdf = provinces[provinces["province_code"] == "NP04"]

loose = gpd.clip(districts, gandaki_gdf)
print(len(loose))
print(loose.geom_type.value_counts())

clean = gpd.clip(districts, gandaki_gdf, keep_geom_type=True)
print(len(clean))

In [ ]:
world = gpd.read_file(DATA / "world_countries.geojson")
region = gpd.clip(world, box(78, 25, 90, 31))
print(list(region["name"]))

fig, ax = plt.subplots(figsize=(8, 5))
region.plot(ax=ax, color="lightgrey", edgecolor="black", linewidth=0.5)
region[region["name"] == "Nepal"].plot(ax=ax, color="tab:red")
ax.set_title("Nepal and its neighbours, clipped to a box")
plt.show()

The plain clip returns 20 rows: 11 polygons, which are Gandaki's districts, and 9 `MultiLineString` slivers from the districts that only touch Gandaki's border. With `keep_geom_type=True` only the 11 real districts remain.

The world map, clipped to longitude 78°–90°E and latitude 25°–31°N, holds five countries: India, Bangladesh, Bhutan, Nepal and China. The world layer is at a much coarser scale than the district layer, which is fine for a location map.

### Practice

**Q1 · one line:** Clip `slides` to `gandaki_gdf` and count the landslides per district.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
gpd.clip(slides, gandaki_gdf)["district"].value_counts()
```

10 landslides: 5 in Myagdi, 4 in Gorkha and 1 in Kaski. Points cannot become slivers, so no `keep_geom_type` is needed.
</details>

**Q2 · two lines:** Clip the world countries to a wider box, `box(70, 5, 100, 40)`. Print how many countries have a part inside it, and their names.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
wide = gpd.clip(world, box(70, 5, 100, 40))
print(len(wide), list(wide["name"]))
```

13 countries, from Sri Lanka in the south to Kyrgyzstan in the north. The larger the box, the more neighbours appear: choose the box to suit the story of your map.
</details>

**Q3 · three lines:** Clip the districts to Bagmati province (`NP03`) with `keep_geom_type=True`. Print how many districts you get and their geometry types, then draw them.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
bagmati_gdf = provinces[provinces["province_code"] == "NP03"]
bagmati_districts = gpd.clip(districts, bagmati_gdf, keep_geom_type=True)
print(len(bagmati_districts), bagmati_districts.geom_type.value_counts())
bagmati_districts.plot(edgecolor="white")
plt.show()
```

13 districts, all polygons, the same number you found by attribute in B5. When a selection by location and a selection by attribute agree, both are probably right.
</details>

## 7. Selecting or clipping? Know the difference

Both give you "the features in an area", but they answer different questions:

| | Select by location | Clip |
| --- | --- | --- |
| Code | `districts[districts.intersects(area)]` | `gpd.clip(districts, area)` |
| Shapes | whole, unchanged | cut at the edge of `area` |
| Good for | "**Which** districts are affected?" | "**How much** of each district is inside?" |

A selection and a clip with the same area keep the same rows, so they keep the same row labels. When you divide one by the other, pandas matches the rows by label, as in B5. That gives the share of each district inside the area.

**Predict:** both keep 8 districts for the valley box. Will their total areas be the same?

In [ ]:
selected = districts[districts.intersects(valley_box)]
clipped = gpd.clip(districts, valley_box)
print(len(selected), len(clipped))

whole_km2 = selected.to_crs(32645).area / 1000000     # each whole district
inside_km2 = clipped.to_crs(32645).area / 1000000     # the part of each district inside the box
print(round(whole_km2.sum()), round(inside_km2.sum()))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
selected.plot(ax=ax1, color="tab:blue", edgecolor="white")
ax1.set_title("Selected: whole districts")
clipped.plot(ax=ax2, color="tab:orange", edgecolor="white")
ax2.set_title("Clipped: cut to the box")
plt.show()

Both keep 8 districts, but the selection covers about 10,360 km² and the clip only about 1,033 km². The selection keeps all of Dhading and Nuwakot, though only a corner of each reaches the box.

The two maps show it at a glance: on the left, whole districts spread far beyond the valley; on the right, a rectangle. `plt.subplots(1, 2)` makes two axes side by side, as in A10.

### Practice

**Q1 · one line:** Use `inside_km2` and `whole_km2` from the example. What share of the selected area lies inside the box? Print it in percent, rounded to 1 decimal place.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(round(inside_km2.sum() / whole_km2.sum() * 100, 1))
```

About 10.0%. Nine tenths of the selected area lies outside the valley box.
</details>

**Q2 · two lines:** How many km² of the selected districts lie **outside** the box? Store the answer in `outside_km2` and print it rounded to a whole number.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
outside_km2 = whole_km2.sum() - inside_km2.sum()
print(round(outside_km2))
```

About 9,328 km² lie outside the box. A selection can hide a lot of area you did not ask for.
</details>

**Q3 · three lines:** Divide `inside_km2` by `whole_km2` to get the percentage of each district inside the box. Put it in a table with the district names from `selected`, sorted from largest share. Both Series have the same 8 row labels, in a different order, because `gpd.clip` does not keep the original row order.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
share = (inside_km2 / whole_km2 * 100).round(1)
result = pd.DataFrame({"district": selected["district"], "share_pct": share})
result.sort_values("share_pct", ascending=False)
```

Bhaktapur is 100.0% inside, Kathmandu 98.7% and Lalitpur 40.9%, while Dhading and Sindhupalchok have only 1.4%. The division matched rows by label, not by position, so the order difference did not matter. Kathmandu is not quite 100%: the box is approximate, not the real valley.
</details>

## 8. When it breaks: an empty selection

Selecting one shape with `.iloc[0]` is the pattern of this lesson. The cell below has a small typo in the district name. Run it.

In [ ]:
kathmandu = districts.loc[districts["district"] == "Katmandu", "geometry"].iloc[0]

The last line reads: `IndexError: single positional indexer is out-of-bounds`.

`IndexError` (A3) means "there is no item at that position". No district is called `Katmandu`, so the mask is all `False` and the selection is empty. `.iloc[0]` then asks for the first item of nothing.

When you see this error after `.iloc[0]`, check the name: spelling, capitals, and the dataset's own spelling (Chitawan, not Chitwan). `.str.contains` from B5 helps you find it.

**Try it:** fix the name so the cell runs.

### A quieter problem: two different CRSs

Some mistakes raise no error at all. Below, Dolakha's shape is converted to UTM 45N (metres), but the landslides are still in degrees. **Predict:** how many landslides will be found inside Dolakha?

In [ ]:
dolakha_utm = districts[districts["district"] == "Dolakha"].to_crs(32645).geometry.iloc[0]
print(slides.within(dolakha_utm).sum())

The answer is 0, not 9, and there is no warning. A shapely shape carries no CRS, so GeoPandas cannot warn you. To Python, Dolakha now lies hundreds of thousands of units away from points whose coordinates are near 86 and 27.

The rule: before any predicate or clip, make sure both layers use the **same CRS**. Print `.crs` for both when a result looks wrong.

## Mixed review

These exercises combine everything in this lesson. Each one needs 5 to 10 lines. Plan in words first, then write the code.

### Review 1 · Is the whole trek in Solukhumbu?

Read `everest_trek.csv` and make its 8 stops into points. Take the shape of Solukhumbu and count how many stops lie within it. Then clip the districts to a box around the trek, `box(86.4, 27.5, 87.2, 28.2)`, and draw the clipped districts with the trek stops on top.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
trek_table = pd.read_csv(DATA / "everest_trek.csv")
trek = gpd.GeoDataFrame(trek_table,
                        geometry=gpd.points_from_xy(trek_table["lon"], trek_table["lat"]),
                        crs="EPSG:4326")
solukhumbu = districts.loc[districts["district"] == "Solukhumbu", "geometry"].iloc[0]
print(trek.within(solukhumbu).sum(), "of", len(trek))

fig, ax = plt.subplots(figsize=(6, 6))
gpd.clip(districts, box(86.4, 27.5, 87.2, 28.2)).plot(ax=ax, column="district", legend=True, edgecolor="white",
                                                      legend_kwds={"loc": "upper left", "bbox_to_anchor": (1, 1)})
trek.plot(ax=ax, color="black", markersize=15)
ax.set_title("Lukla to Everest Base Camp: all stops in Solukhumbu")
plt.show()
```

All 8 stops, from Lukla to Everest Base Camp, lie within Solukhumbu. Unlike the Everest summit point, the trek stops are well inside the border.
</details>

### Review 2 · Bagmati districts on the Gandaki border

Take the shape of Gandaki province. Find the districts of Bagmati (`NP03`) that touch it, by combining an attribute mask and a predicate mask. Print their names, then map them in orange over all Bagmati districts in light grey.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
gandaki = provinces.loc[provinces["province_code"] == "NP04", "geometry"].iloc[0]
in_bagmati = districts["province_code"] == "NP03"
border = districts[in_bagmati & districts.touches(gandaki)]
print(list(border["district"]))

fig, ax = plt.subplots(figsize=(7, 5))
districts[in_bagmati].plot(ax=ax, color="lightgrey", edgecolor="white")
border.plot(ax=ax, color="tab:orange", edgecolor="white")
ax.set_title("Bagmati districts on the border with Gandaki")
plt.show()
```

Two districts, Chitawan and Dhading, share a border with Gandaki. A district touches a province when it shares part of the province's outline, so `touches` works with shapes of any size.
</details>

### Review 3 · The landslides of Bagmati

Clip `slides` to Bagmati province. Print how many landslides are inside and how many per district. Then draw the Bagmati districts clipped with `keep_geom_type=True`, with the landslides on top.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
bagmati_gdf = provinces[provinces["province_code"] == "NP03"]
bagmati_slides = gpd.clip(slides, bagmati_gdf)
print(len(bagmati_slides))
print(bagmati_slides["district"].value_counts())

fig, ax = plt.subplots(figsize=(8, 5))
gpd.clip(districts, bagmati_gdf, keep_geom_type=True).plot(ax=ax, color="lightgrey", edgecolor="white")
bagmati_slides.plot(ax=ax, color="black", markersize=12)
ax.set_title("Practice landslides in Bagmati province")
plt.show()
```

26 landslides: 9 in Dolakha, 9 in Chitawan, 5 in Rasuwa and 3 in Sindhupalchok. That matches the 26 Bagmati records of the CSV's `province` column, which you counted in B6.
</details>

### Review 4 · Which provinces touch Bagmati?

Read `province_names.csv` and join the names onto `provinces` (B6). Take Bagmati's shape, select the provinces that touch it, and print their names. Then map them, with Bagmati in red.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
names = pd.read_csv(DATA / "province_names.csv")
provinces_named = provinces.merge(names, on="province_code")
bagmati = provinces_named.loc[provinces_named["province"] == "Bagmati", "geometry"].iloc[0]
touching = provinces_named[provinces_named.touches(bagmati)]
print(list(touching["province"]))

fig, ax = plt.subplots(figsize=(9, 5))
provinces_named.plot(ax=ax, color="lightgrey", edgecolor="white")
touching.plot(ax=ax, column="province", legend=True, edgecolor="white")
provinces_named[provinces_named["province"] == "Bagmati"].plot(ax=ax, color="tab:red")
ax.set_title("Bagmati (red) and the provinces that touch it")
plt.show()
```

Bagmati touches Koshi, Madhesh and Gandaki. It does not reach Lumbini, Karnali or Sudurpashchim. Joining, selecting by attribute and selecting by location work together: each answer is a GeoDataFrame you can map.
</details>

## Recap

- A **spatial predicate** answers yes or no about two shapes: `within`, `contains`, `intersects`, `touches`. A point on an edge is not `within`; `intersects` counts it.
- On a GeoDataFrame, a predicate asks every row and gives a mask: `districts[districts.touches(kathmandu)]`. Take one shape out with `.loc[mask, "geometry"].iloc[0]`.
- Points near borders may fall outside every polygon, like the Everest summit point with simplified boundaries. Count the matches to check.
- `gpd.clip(layer, mask)` cuts lines and polygons at the mask's edge. Use `keep_geom_type=True` to drop the line slivers from neighbours that only touch the mask.
- Select when you want whole features ("which?"), clip when you want the part inside ("how much?"). Both layers must use the same CRS, or results can be silently wrong.

### Words you learned

| Word | Meaning |
| --- | --- |
| spatial predicate | A yes-or-no question about two shapes, such as `within` or `touches` |
| `within` / `contains` | One shape lies completely inside the other; the same question from opposite sides |
| `intersects` | The two shapes share at least one point |
| `touches` | The shapes share edge points, but their insides do not overlap |
| disjoint | Sharing no point at all; the opposite of `intersects` |
| neighbour | A shape that touches another, such as districts that share a border |
| select by location | Keep whole features chosen by a spatial predicate |
| clip | Cut a layer to the shape of a mask, keeping only the parts inside |
| mask | In clipping, the shape used as the cookie-cutter |
| `box()` | Makes a rectangle from `x_min, y_min, x_max, y_max` |
| `keep_geom_type=True` | Keeps only clip results of the input's geometry type |
| sliver | A thin, unwanted piece of geometry, such as a line along a shared border |
| location map | A small map showing where the study area lies in a larger region |

## Next

**B8 · Buffers and nearest features:** draw zones of a set distance around points and lines, and find the nearest district headquarters to every landslide, measured in kilometres.

<!-- pygile-nav -->
**PyGILE Course** · lesson 17 of 21 · [Course home](../README.md)  
← Previous: [B6 · Joins and dissolve: names, mismatches and provinces](B06_Joins_and_dissolve.ipynb) · Next: [B8 · Buffers and nearest features: how close is close?](B08_Buffers_and_nearest_features.ipynb) →